# Misread detection on Kaggle

Does our reader's token confidence find the numbers a transcriber misread?

**Settings:** Accelerator *GPU T4 x2*, Internet *on*. **Add Input:** the 3 private datasets (`edu_text`, page images, `qwen3b_ft` + `reader_lora`). Folder names don't matter, cell 2 finds them.

**Outputs:** per run folder `results/detect/<run>/`: `scores.csv` (confidence of every maths token), `labels.csv` (misread = 1 where the expert disagrees), `detection_metrics.json` (how many misreads the top-k flags catch), `verify_metrics.json` (A/B check). Only the `*_metrics.json` files may leave Kaggle: the CSVs quote student work.

## 1. Code + packages

In [ ]:
# embedded project code (generated by scripts/make_detect_notebook.py; regenerate after code changes)
# files: src/detect/labels.py, src/detect/score_tokens.py, src/detect/evaluate_detection.py, src/detect/verify_choice.py, src/detect/test_detect.py, src/eval/run_ocr.py, src/eval/evaluate_recognition.py, src/eval/llm.py, src/eval/edu_compat.py, src/eval/items_with_expert_transcript.csv, src/eval/prompts/Initial_prompt_wo_problem_statement_v6.txt
CODE_ZIP = '''
...
'''

In [ ]:
import os, sys, glob, json, shutil, subprocess, zipfile, base64, io
IN = os.environ.get('KS_INPUT', '/kaggle/input')
WORK = os.environ.get('KS_WORK', '/kaggle/working')
REPO = os.environ.get('KS_REPO') or f'{WORK}/ci'

def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=REPO)
    if r.returncode: raise RuntimeError(f'failed ({r.returncode}): {cmd}')

# the code is embedded below (src/detect + the src/eval modules it imports), no git needed
if not os.environ.get('KS_REPO'):
    zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP))).extractall(REPO)
print(sorted(glob.glob(f'{REPO}/src/*/*.py')))
if os.path.exists('/kaggle'):
    sh('pip install -q -U "transformers>=4.51" accelerate peft')

## 2. Find the data

In [ ]:
roots = [IN, f'{WORK}/_in']

def find(pattern, up=1, required=True):
    # shallowest match first; a recursive glob over /kaggle/input walks every file and takes ~15 min
    hits = []
    for depth in range(7):
        hits = sorted(h for r in roots for h in glob.glob(f'{r}/' + '*/' * depth + pattern))
        if hits: break
    if not hits and not required: return None
    if not hits: raise FileNotFoundError(f'nothing matches {pattern} under {roots}')
    p = hits[0]
    for _ in range(up): p = os.path.dirname(p)
    return p

# Kaggle usually unzips uploads; if it kept a .zip, extract it
if find('Rectified_recognized_markdown_done_Anon', required=False) is None:
    for depth in range(4):
        for z in glob.glob(f'{IN}/' + '*/' * depth + '*.zip'):
            zipfile.ZipFile(z).extractall(f'{WORK}/_in/' + os.path.basename(z)[:-4])
EDU = find('Rectified_recognized_markdown_done_Anon')                  # expert transcripts
IMG = find('Screenshot_output_anon/Homework*/student_*/*.png', up=4)    # page images
# optional: without them only Gemini is scored, with the untuned base reader
QWEN = find('qwen3b_ft', up=0, required=False)                          # our 3B transcripts
LORA = os.environ.get('KS_LORA') or find('reader_lora', up=0, required=False)  # fine-tuned reader
GEMINI = f'{EDU}/Observationset_Final/v6_Gemini_2p5'
for k, v in dict(EDU=EDU, IMG=IMG, QWEN=QWEN, LORA=LORA, GEMINI=GEMINI).items():
    print(f'{k:6s} {v}  exists={bool(v) and os.path.isdir(v)}')
print(len(glob.glob(f'{IMG}/Screenshot_output_anon/Homework*/student_*/*.png')), 'page images')

## 3. Settings

`LIMIT = None` scores all 293 pages (~8 s/page on a T4, ~40 min per transcriber); `LIMIT = 5` is a smoke test. Use *Save Version -> Run All* so it runs in the background.

In [ ]:
LIMIT = None
K = 0                                           # A/B check of the top-K numbers per page (0 = skip)
READER = os.environ.get('KS_READER', 'Qwen/Qwen2.5-VL-3B-Instruct')
PROMPT = 'src/eval/prompts/Initial_prompt_wo_problem_statement_v6.txt'
OUT = f'{WORK}/results/detect'
# (name, transcripts, model dir, use the fine-tuned adapter?)
FT = bool(LORA) and not os.environ.get('KS_NO_ADAPTER')
tag = 'ftreader' if FT else 'basereader'
RUNS = ([(f'qwen3b_ft_{tag}', QWEN, 'qwen3b_ft', FT)] if QWEN else []) + \
       [(f'gemini25_{tag}', GEMINI, 'gemini-2.5-pro', FT)]
# ablation (does fine-tuning help detection?): add ('qwen3b_ft_basereader', QWEN, 'qwen3b_ft', False)
print('runs:', [r[0] for r in RUNS])

## 4. Run: labels (CPU) -> token scores (GPU) -> metrics -> A/B check

In [ ]:
sh('python src/detect/test_detect.py')
lim = f'--limit {LIMIT}' if LIMIT else ''
for name, root, mdir, ft in RUNS:
    d, ad = f'{OUT}/{name}', (f'--adapter {LORA}' if ft else '')
    common = f'--edu-data {EDU} --target-root {root} --model-dir {mdir}'
    print(f'\n===== {name}')
    sh(f'python src/detect/labels.py {common} --out {d}')
    sh(f'python src/detect/score_tokens.py {common} --images {IMG} --prompt-file {PROMPT} --out {d} '
       f'--reader hf:{READER} {ad} {lim}')
    sh(f'python src/detect/evaluate_detection.py --dir {d}')
    if K:
        sh(f'python src/detect/verify_choice.py --dir {d} {common} --images {IMG} '
           f'--reader hf:{READER} --k {K} {ad} {lim}')

## 5. Results

**recall@3** = share of all misreads caught by flagging the 3 least confident numbers per page; **precision@3** = share of flags that are real misreads; beat the `random` row.

In [ ]:
import pandas as pd
for name, *_ in RUNS:
    d = f'{OUT}/{name}'
    m = json.load(open(f'{d}/detection_metrics.json'))
    print(f"\n===== {name}: {m['pages']} pages, {m['numbers']} numbers, {m['misreads']} misreads "
          f"(base rate {m['base_rate']})")
    t = pd.DataFrame(m['signals']).T
    t['caught@3'] = (pd.to_numeric(t['recall@3']) * m['misreads']).round().astype(int)
    print(t[['auroc', 'auroc_95CI', 'avg_precision', 'precision@3', 'recall@3', 'caught@3']].to_string())
    if os.path.exists(f'{d}/verify_metrics.json'):
        print('A/B check:', json.load(open(f'{d}/verify_metrics.json')))

## 6. Download the metrics (no student text)

Output tab -> `detect_metrics.zip`.

In [ ]:
with zipfile.ZipFile(f'{WORK}/detect_metrics.zip', 'w') as z:
    for f in glob.glob(f'{OUT}/*/*_metrics.json') + glob.glob(f'{OUT}/*/labels_summary.json'):
        z.write(f, os.path.relpath(f, OUT))
print(zipfile.ZipFile(f'{WORK}/detect_metrics.zip').namelist())